# সহজ ভাষায় Notebook Guide

এই notebook **সম্পূর্ণ self-contained** — project folder, GPU, Docker daemon বা cloud account কিছুই লাগে না। চারটা হাতিয়ার বানাব: একটা VRAM estimator (কোন quantization কোন GPU-তে আঁটে), GPU আসলে ব্যবহার হচ্ছে কিনা বোঝার parser (`nvidia-smi`, `ollama ps`), একটা মাসিক খরচের calculator, আর একটা deployment readiness checklist। Technical term English-এ, explanation Bangla-তে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন। দরকার শুধু `pandas`।
2. **Verification-এর সৎ কথা:** এই notebook যে machine-এ চালানো হয়েছে সেখানে GPU বা cloud account ছিল না। তাই —
   * VRAM-এর সংখ্যা **estimate** (formula থেকে), মাপা না। নিজের GPU-তে `nvidia-smi` আর `ollama ps` দিয়ে মেলান।
   * Section 2-এর `nvidia-smi` / `ollama ps` output **illustrative sample**, হাতে লেখা।
   * Section 3-এর দাম **ইচ্ছাকৃতভাবে নকল** (`FAKE_RATES`) — provider-এর pricing page থেকে আজকের সংখ্যা বসিয়ে নিজে চালান।

# GPU Passthrough & Cloud Deployment — Lecture Notebook

1. VRAM estimator: weights + KV cache + overhead
2. GPU কি সত্যিই কাজ করছে? `nvidia-smi` আর `ollama ps` পড়া
3. মাসিক খরচ: GPU VM, CPU VM, hosted API (নকল দাম)
4. Deployment readiness checklist

In [1]:
import re
import pandas as pd
from IPython.display import Markdown, display
print("pandas", pd.__version__)

pandas 3.1.0


## 1. VRAM Estimator

GPU-তে model চালাতে তিনটা জিনিস VRAM-এ বসে:

* **Weights** = parameter সংখ্যা × প্রতি weight-এ bit ÷ 8। Quantization এই bit কমায়: FP16 = 16, Q8_0 ≈ 8.5, Q4_K_M ≈ 4.85 (llama.cpp-র block format-এ scale-এর জন্য কিছু বাড়তি bit; Q4_K_M কিছু tensor-এ 6-bit রাখে)।
* **KV cache** = 2 (K আর V) × layer × KV head × head dim × byte × context token। Llama 3.1 8B-এ ৩২ layer, ৮টা KV head (grouped-query attention — ৩২টা query head ৮টা KV head ভাগ করে), head dim ১২৮। Context যত লম্বা, KV তত বড় — model-এর আকারের সাথে সম্পর্ক নেই।
* **Overhead** = CUDA context + compute buffer। এখানে ধরে নিচ্ছি ০.৮ GB — এটা একটা **অনুমান**; আসল সংখ্যা `nvidia-smi`-তে মাপুন।

Architecture সংখ্যা model-এর public `config.json` থেকে। সব GB = 10⁹ byte (Ollama-র library page-এর মতো)।

In [2]:
ARCH = {   # from each model's public config.json
    "llama3.1:8b": {"params_b": 8.03, "layers": 32, "kv_heads": 8, "head_dim": 128},
    "llama3.2:3b": {"params_b": 3.21, "layers": 28, "kv_heads": 8, "head_dim": 128},
}
BITS = {"Q4_K_M": 4.85, "Q8_0": 8.5, "FP16": 16.0}   # effective bits per weight (approximate)
OVERHEAD_GB = 0.8                                    # assumption: CUDA context + compute buffers

def weights_gb(model, quant):
    return ARCH[model]["params_b"] * BITS[quant] / 8

def kv_bytes_per_token(model, kv_bytes=2):          # kv_bytes: 2 = f16 cache, 1 = q8_0 cache
    a = ARCH[model]
    return 2 * a["layers"] * a["kv_heads"] * a["head_dim"] * kv_bytes

def vram_gb(model, quant, ctx, kv_bytes=2):
    return weights_gb(model, quant) + kv_bytes_per_token(model, kv_bytes) * ctx / 1e9 + OVERHEAD_GB

assert kv_bytes_per_token("llama3.1:8b") == 128 * 1024                # 128 KiB per token
assert abs(weights_gb("llama3.1:8b", "Q4_K_M") - 4.9) < 0.1           # Ollama lists the default tag at 4.9 GB

rows = []
for model, quant, ctx, kv in [("llama3.1:8b", q, c, 2) for q in BITS for c in (4096, 8192, 32768)] + \
                             [("llama3.1:8b", "Q4_K_M", 32768, 1), ("llama3.2:3b", "Q4_K_M", 8192, 2)]:
    rows.append({"model": model, "quant": quant, "context": ctx, "kv_cache": "f16" if kv == 2 else "q8_0",
                 "weights_GB": round(weights_gb(model, quant), 1),
                 "kv_GB": round(kv_bytes_per_token(model, kv) * ctx / 1e9, 2),
                 "total_GB_est": round(vram_gb(model, quant, ctx, kv), 1)})
vram = pd.DataFrame(rows)
vram

,model,quant,context,kv_cache,weights_GB,kv_GB,total_GB_est
0,llama3.1:8b,Q4_K_M,4096,f16,4.9,0.54,6.2
1,llama3.1:8b,Q4_K_M,8192,f16,4.9,1.07,6.7
2,llama3.1:8b,Q4_K_M,32768,f16,4.9,4.29,10.0
3,llama3.1:8b,Q8_0,4096,f16,8.5,0.54,9.9
4,llama3.1:8b,Q8_0,8192,f16,8.5,1.07,10.4
5,llama3.1:8b,Q8_0,32768,f16,8.5,4.29,13.6
6,llama3.1:8b,FP16,4096,f16,16.1,0.54,17.4
7,llama3.1:8b,FP16,8192,f16,16.1,1.07,17.9
8,llama3.1:8b,FP16,32768,f16,16.1,4.29,21.2
9,llama3.1:8b,Q4_K_M,32768,q8_0,4.9,2.15,7.8


**পড়ার নিয়ম:**

* Q4_K_M-এ weight ≈৪.৯ GB — Ollama-র default `llama3.1:8b` tag-এর download size-এর সাথে মেলে, তাই formula ঠিক পথে।
* ৪k থেকে ৩২k context-এ KV ০.৫ থেকে ৪.৩ GB — **লম্বা context একটা ছোট model-এর weight-এর সমান VRAM খায়।** RAG-এ top-k × chunk যত বড়, `num_ctx` তত বড় লাগে, VRAM তত বেশি।
* KV cache q8_0-এ (Ollama-র `OLLAMA_KV_CACHE_TYPE=q8_0`, flash attention চালু থাকলে) KV অর্ধেক — লম্বা context-এ কাজের knob।
* FP16 ≈১৮ GB — ১৬ GB-এর GPU-তে আঁটে না।

এবার কয়েকটা GPU-র সাথে মেলাই। Rule of thumb: VRAM-এর ৯০% পর্যন্ত plan করুন (driver, display, fragmentation)। না আঁটলে Ollama model-কে CPU আর GPU-তে ভাগ করে (partial offload) — চলে, কিন্তু speed CPU-র দিকে নেমে যায়।

In [3]:
GPUS_GB = {"laptop GPU 8 GB": 8, "RTX 3060 12 GB": 12, "T4 16 GB": 16, "L4 / A10G 24 GB": 24, "A100 40 GB": 40}
USABLE = 0.9

def placement(need_gb, vram):
    return "GPU" if need_gb <= USABLE * vram else "split" if need_gb <= 2 * vram else "CPU"

configs = [("llama3.1:8b", "Q4_K_M", 8192), ("llama3.1:8b", "Q4_K_M", 32768), ("llama3.1:8b", "Q8_0", 8192),
           ("llama3.1:8b", "FP16", 8192), ("llama3.2:3b", "Q4_K_M", 8192)]
fit = pd.DataFrame({f"{m} {q} @{c//1024}k ({vram_gb(m, q, c):.1f} GB)": {g: placement(vram_gb(m, q, c), v) for g, v in GPUS_GB.items()}
                    for m, q, c in configs}).T
assert fit.loc[fit.index[0], "laptop GPU 8 GB"] == "GPU" and fit.loc[fit.index[3], "T4 16 GB"] == "split"
fit

,laptop GPU 8 GB,RTX 3060 12 GB,T4 16 GB,L4 / A10G 24 GB,A100 40 GB
llama3.1:8b Q4_K_M @8k (6.7 GB),GPU,GPU,GPU,GPU,GPU
llama3.1:8b Q4_K_M @32k (10.0 GB),split,GPU,GPU,GPU,GPU
llama3.1:8b Q8_0 @8k (10.4 GB),split,GPU,GPU,GPU,GPU
llama3.1:8b FP16 @8k (17.9 GB),CPU,split,split,GPU,GPU
llama3.2:3b Q4_K_M @8k (3.7 GB),GPU,GPU,GPU,GPU,GPU


**পড়ার নিয়ম:** Course-এর default (Q4_K_M, ৮k context) ৮ GB laptop GPU-তে টায়টায় আঁটে, ১২ GB-তে আরামে। ৩২k context দিলেই ৮ GB-তে "split"। FP16 শুধু ২৪ GB-এ। এক ৮B model-এর জন্য A100 ভাড়া করা মানে ৮০% VRAM অলস — খরচের section-এ এর দাম দেখব। `split` মানে "চলবে না" না — মানে ধীর; কতটা ধীর সেটা নিজের machine-এ `ollama run llama3.1:8b --verbose`-এর `eval rate` দিয়ে মাপুন।

## 2. GPU কি সত্যিই কাজ করছে?

GPU-র সবচেয়ে বিপজ্জনক ব্যর্থতা **নীরব**: device request ভুলে গেলে Ollama কোনো error ছাড়াই CPU-তে চলে, শুধু দশ গুণ ধীর। প্রমাণ দুই ধাপে:

1. Container GPU দেখে কিনা: `docker compose exec ollama nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader`
2. Model GPU-তে load হয়েছে কিনা: `docker compose exec ollama ollama ps` → `PROCESSOR` column।

Ollama-র docs (October 2026) বলে NVIDIA-তে compute capability 5.0+ আর driver 550+ লাগে; compute capability 5.0-6.2-এর card-এ driver 570+। নিচের output-গুলো **illustrative sample** — হাতে লেখা, কোনো আসল GPU থেকে না।

In [4]:
NVIDIA_SMI = {   # illustrative samples of the CSV query above
    "T4 cloud VM": "Tesla T4, 7.5, 15360 MiB, 550.54.15",
    "old desktop": "GeForce GTX 1060 6GB, 6.1, 6144 MiB, 535.183.01",
    "toolkit missing": "",
}

def check_gpu(csv_line):
    if not csv_line.strip():
        return {"gpu": None, "ok": False, "why": "no GPU visible in the container: toolkit/runtime or device request missing"}
    name, cc, mem, driver = [x.strip() for x in csv_line.split(",")]
    cc, major = float(cc), int(driver.split(".")[0])
    need = 570 if 5.0 <= cc <= 6.2 else 550
    if cc < 5.0:
        return {"gpu": name, "ok": False, "why": f"compute capability {cc} < 5.0: not supported, Ollama falls back to CPU"}
    ok = major >= need
    return {"gpu": name, "vram": mem, "ok": ok, "why": "ok" if ok else f"driver {major} < {need}: upgrade the HOST driver"}

pd.DataFrame([{"case": k} | check_gpu(v) for k, v in NVIDIA_SMI.items()])

,case,gpu,vram,ok,why
0,T4 cloud VM,Tesla T4,15360 MiB,True,ok
1,old desktop,GeForce GTX 1060 6GB,6144 MiB,False,driver 535 < 570: upgrade the HOST driver
2,toolkit missing,NaN,NaN,False,no GPU visible in the container: toolkit/runti...


In [5]:
HEADER = "NAME           ID              SIZE      PROCESSOR          CONTEXT    UNTIL"

def ollama_ps(size, processor, ctx):   # illustrative `ollama ps` output, same column layout
    return f"{HEADER}\nllama3.1:8b    a1b2c3d4e5f6    {size:<8}  {processor:<17}  {ctx:<9}  29 minutes from now"

SAMPLES = {"GPU VM, override applied": ollama_ps("6.7 GB", "100% GPU", 8192),
           "8 GB GPU, context 32k": ollama_ps("9.9 GB", "27%/73% CPU/GPU", 32768),
           "GPU VM, override forgotten": ollama_ps("6.7 GB", "100% CPU", 8192)}
print(SAMPLES["8 GB GPU, context 32k"])

def diagnose(ps_text):
    out = []
    for line in ps_text.strip().splitlines()[1:]:
        name, _id, size, processor, ctx = re.split(r"\s{2,}", line.strip())[:5]
        if processor == "100% GPU":
            why = "fully on GPU"
        elif processor == "100% CPU":
            why = "GPU not used: check -f docker-compose.gpu.yml, then nvidia-smi inside the container"
        else:
            why = "partial offload: lower num_ctx, KV cache q8_0, smaller quant, or bigger GPU"
        out.append({"model": name, "size": size, "processor": processor, "context": ctx, "diagnosis": why})
    return out

pd.DataFrame([{"case": k} | d for k, v in SAMPLES.items() for d in diagnose(v)])

NAME           ID              SIZE      PROCESSOR          CONTEXT    UNTIL
llama3.1:8b    a1b2c3d4e5f6    9.9 GB    27%/73% CPU/GPU    32768      29 minutes from now


,case,model,size,processor,context,diagnosis
0,"GPU VM, override applied",llama3.1:8b,6.7 GB,100% GPU,8192,fully on GPU
1,"8 GB GPU, context 32k",llama3.1:8b,9.9 GB,27%/73% CPU/GPU,32768,"partial offload: lower num_ctx, KV cache q8_0,..."
2,"GPU VM, override forgotten",llama3.1:8b,6.7 GB,100% CPU,8192,"GPU not used: check -f docker-compose.gpu.yml,..."


**পড়ার নিয়ম:** `nvidia-smi` host-এ চললেই হবে না — container-এর **ভেতরে** চলতে হবে (Container Toolkit `utility` capability দিয়ে binary-টা ঢোকায়)। আর container GPU দেখলেও model-টা GPU-তে বসেছে কিনা `ollama ps` বলে। "override forgotten" case-টাই সবচেয়ে সাধারণ: VM-এ GPU, driver, toolkit সব ঠিক, কিন্তু `docker compose up -d` চালানো হয়েছে `-f docker-compose.gpu.yml` ছাড়া।

## 3. মাসিক খরচ: GPU VM, CPU VM, Hosted API

⚠️ **নিচের সব দাম নকল** (`FAKE_RATES`) — শুধু হিসাবের পদ্ধতি দেখানোর জন্য গোল সংখ্যা। কোনো provider-এর আসল দাম না। আপনার provider-এর pricing page থেকে আজকের সংখ্যা (আর BDT-তে চাইলে আজকের exchange rate) বসিয়ে আবার চালান।

একটা VM-এর bill = চালু থাকার ঘণ্টা × ঘণ্টার দাম + disk (VM বন্ধ থাকলেও) + বাইরে যাওয়া data (egress)। Hosted API-র bill = token × token-এর দাম।

In [6]:
FAKE_RATES = {                      # FAKE example numbers. Replace every one before deciding anything.
    "gpu_vm_per_hour": 1.00, "cpu_vm_per_hour": 0.10,
    "disk_per_gb_month": 0.10, "egress_per_gb": 0.10,
    "api_per_1m_input_tokens": 1.00, "api_per_1m_output_tokens": 3.00,
}
HOURS_PER_MONTH = 730

def vm_month(per_hour, hours, disk_gb=100, egress_gb=20, r=FAKE_RATES):
    return per_hour * hours + disk_gb * r["disk_per_gb_month"] + egress_gb * r["egress_per_gb"]

def api_month(questions_per_day, in_tokens=1500, out_tokens=250, r=FAKE_RATES):
    q = questions_per_day * 30                        # RAG prompts are long: retrieved passages count as input
    return q * (in_tokens * r["api_per_1m_input_tokens"] + out_tokens * r["api_per_1m_output_tokens"]) / 1e6

office_hours = 10 * 22                                # VM stopped nights/weekends; disk is still billed
scenarios = {
    "GPU VM 24/7": vm_month(FAKE_RATES["gpu_vm_per_hour"], HOURS_PER_MONTH),
    "GPU VM office hours (220 h)": vm_month(FAKE_RATES["gpu_vm_per_hour"], office_hours),
    "CPU VM 24/7 (llama3.2:3b, slow)": vm_month(FAKE_RATES["cpu_vm_per_hour"], HOURS_PER_MONTH),
    "hosted API, 200 questions/day": api_month(200),
    "hosted API, 2000 questions/day": api_month(2000),
}
print("FAKE RATES - illustration only")
pd.DataFrame({"fake_cost_per_month": {k: round(v, 2) for k, v in scenarios.items()}})

FAKE RATES - illustration only


,fake_cost_per_month
GPU VM 24/7,742.0
GPU VM office hours (220 h),232.0
"CPU VM 24/7 (llama3.2:3b, slow)",85.0
"hosted API, 200 questions/day",13.5
"hosted API, 2000 questions/day",135.0


In [7]:
per_question = api_month(1) / 30
breakeven_per_day = scenarios["GPU VM 24/7"] / per_question / 30
print(f"hosted API cost per question (fake rates): {per_question:.5f}")
print(f"a 24/7 GPU VM costs the same as ~{breakeven_per_day:,.0f} questions/day on the API (fake rates)")

# Can one GPU even serve that? Little's law from Week 15: one Ollama slot, W seconds per answer.
W_GPU_S = 2.0                                          # assumption: measure yours with the Week 15 benchmark
capacity_per_day = 24 * 3600 / W_GPU_S
print(f"one slot at W={W_GPU_S}s: max ~{capacity_per_day:,.0f} answers/day at 100% busy, "
      f"~{0.6 * capacity_per_day:,.0f}/day at a safe 60%")

hosted API cost per question (fake rates): 0.00225
a 24/7 GPU VM costs the same as ~10,993 questions/day on the API (fake rates)
one slot at W=2.0s: max ~43,200 answers/day at 100% busy, ~25,920/day at a safe 60%


**পড়ার নিয়ম (নকল দামেও পদ্ধতি একই):**

* **২৪/৭ GPU VM-এর বেশিরভাগ টাকা অলস ঘণ্টার।** দিনে কয়েকশো প্রশ্নে GPU হয়তো ১%-ও ব্যস্ত না। Office hours-এ চালু রাখলে bill প্রায় তিন ভাগের এক ভাগ (disk তখনো চলে)।
* **Breakeven:** কম traffic-এ hosted API সস্তা; traffic বাড়লে নিজের GPU সস্তা হয় — আর capacity-র হিসাব (Little's law) বলে এক slot কত পর্যন্ত টানতে পারে। কিন্তু Healthcare Assistant-এ সিদ্ধান্ত শুধু টাকার না: hosted API মানে রোগীর প্রশ্ন তৃতীয় পক্ষের server-এ।
* **Token-এর হিসাবে RAG ভারী:** প্রতিটা প্রশ্নে retrieved passage input token — ১৫০০ input বনাম ২৫০ output। Top-k কমালে API bill কমে, নিজের GPU-তে prefill কমে (Week 15)।
* **যা এখানে নেই কিন্তু আসল bill-এ থাকে:** public IP, snapshot/backup storage, domain, আর সবচেয়ে বড়টা — আপনার সময়।

## 4. Deployment Readiness Checklist

VM-এ `docker compose up` চলা মানেই "deployed" না। নিচের checklist একটা deployment-এর তথ্য (`facts`) নিয়ে প্রতিটা প্রশ্নে ✅/❌ আর fix দেয়। Public port-এর তথ্য সরাসরি compose-এর `ports` থেকে বের করা হয় — `127.0.0.1:` দিয়ে bind না করা যেকোনো port internet-এ খোলা, আর Docker-এর published port ufw-কে পাশ কাটায়।

In [8]:
def public_ports(services):
    # ponytail: short-syntax strings only ("8501", "8501:8501", "127.0.0.1:8000:8000"); long syntax / IPv6 not parsed
    found = set()
    for s in services.values():
        for p in s.get("ports", []):
            parts = str(p).split("/")[0].split(":")
            if len(parts) < 3 or parts[0] not in ("127.0.0.1", "localhost"):
                found.add(int(parts[0]) if len(parts) == 1 else int(parts[-2]))   # "8501" alone = random public host port
    return found

def pinned(image):
    tag = image.rsplit("/", 1)[-1].partition(":")[2]
    return tag not in ("", "latest")

CHECKS = [
    ("Only 22/80/443 reachable from the internet", lambda f: public_ports(f["services"]) <= {22, 80, 443},
     "drop ports from ollama/chroma/api; bind debug ports to 127.0.0.1"),
    ("HTTPS via a reverse proxy", lambda f: f["https"], "Caddy: `your.domain { reverse_proxy ui:8501 }`"),
    ("Login in front of the UI (health data!)", lambda f: f["auth"], "Caddy basic_auth at minimum; SSO for real users"),
    ("SSH key-only", lambda f: f["ssh_key_only"], "PasswordAuthentication no"),
    ("Secrets only in a server-side .env (chmod 600)", lambda f: f["secrets_in_env_file"] and not f["secrets_in_image"],
     "move secrets out of the image/git; rotate the leaked ones"),
    ("Every image tag pinned", lambda f: all(pinned(s["image"]) for s in f["services"].values() if "image" in s),
     "pin tags in .env; keep the previous tag for rollback"),
    ("Healthcheck on every long-running service",
     lambda f: all("healthcheck" in s for s in f["services"].values() if s.get("restart") != "no"), "see Class 35"),
    ("Restart policy + Docker enabled at boot", lambda f: f["survives_reboot"],
     "restart: unless-stopped; systemctl enable docker; test with a real reboot"),
    ("Log rotation", lambda f: f["log_rotation"], "logging: json-file, max-size 10m, max-file 3"),
    ("Backup of irreplaceable volumes, restore tested", lambda f: f["restore_tested"], "tar the volume off-VM; practise a restore"),
    ("GPU actually used (GPU VM only)", lambda f: not f["gpu_vm"] or f["ollama_processor"] == "100% GPU",
     "apply docker-compose.gpu.yml; nvidia-smi inside the container"),
    ("External uptime check + metrics", lambda f: f["monitoring"], "uptime probe from outside the VM; scrape /metrics (Week 15)"),
]

def checklist(facts):
    results = [(bool(test(facts)), item, fix) for item, test, fix in CHECKS]
    lines = [f"- {'✅' if ok else '❌'} {item}" + ("" if ok else f" — *fix:* {fix}") for ok, item, fix in results]
    score = sum(ok for ok, _, _ in results)
    return score, f"**{facts['label']}: {score}/{len(results)}**\n\n" + "\n".join(lines)

In [9]:
HC = {"test": ["CMD", "true"]}       # placeholder: the real healthchecks are in the compose file
FIRST_TRY = {
    "label": "first try (copied the laptop setup)",
    "services": {"ollama": {"image": "ollama/ollama:latest", "ports": ["11434:11434"], "restart": "unless-stopped", "healthcheck": HC},
                 "chroma": {"image": "chromadb/chroma:x.y.z", "restart": "unless-stopped", "healthcheck": HC},
                 "api": {"build": ".", "ports": ["8000:8000"], "restart": "unless-stopped", "healthcheck": HC},
                 "ui": {"build": ".", "ports": ["8501:8501"]}},
    "https": False, "auth": False, "ssh_key_only": True, "secrets_in_env_file": True, "secrets_in_image": False,
    "survives_reboot": False, "log_rotation": False, "restore_tested": False,
    "gpu_vm": True, "ollama_processor": "100% CPU", "monitoring": False,
}
READY = FIRST_TRY | {
    "label": "after the checklist",
    "services": {"caddy": {"image": "caddy:x.y.z", "ports": ["80:80", "443:443"], "restart": "unless-stopped", "healthcheck": HC},
                 "ollama": {"image": "ollama/ollama:x.y.z", "restart": "unless-stopped", "healthcheck": HC},
                 "ollama-init": {"image": "ollama/ollama:x.y.z", "restart": "no"},
                 "chroma": {"image": "chromadb/chroma:x.y.z", "restart": "unless-stopped", "healthcheck": HC},
                 "api": {"build": ".", "ports": ["127.0.0.1:8000:8000"], "restart": "unless-stopped", "healthcheck": HC},
                 "ui": {"build": ".", "restart": "unless-stopped", "healthcheck": HC}},
    "https": True, "auth": True, "survives_reboot": True, "log_rotation": True, "restore_tested": True,
    "ollama_processor": "100% GPU", "monitoring": True,
}
print("public ports, first try:", sorted(public_ports(FIRST_TRY["services"])), "| after:", sorted(public_ports(READY["services"])))
s1, md1 = checklist(FIRST_TRY)
s2, md2 = checklist(READY)
assert s2 == len(CHECKS) and s1 < s2
display(Markdown(md1 + "\n\n" + md2))

public ports, first try: [8000, 8501, 11434] | after: [80, 443]


**first try (copied the laptop setup): 2/12**

- ❌ Only 22/80/443 reachable from the internet — *fix:* drop ports from ollama/chroma/api; bind debug ports to 127.0.0.1
- ❌ HTTPS via a reverse proxy — *fix:* Caddy: `your.domain { reverse_proxy ui:8501 }`
- ❌ Login in front of the UI (health data!) — *fix:* Caddy basic_auth at minimum; SSO for real users
- ✅ SSH key-only
- ✅ Secrets only in a server-side .env (chmod 600)
- ❌ Every image tag pinned — *fix:* pin tags in .env; keep the previous tag for rollback
- ❌ Healthcheck on every long-running service — *fix:* see Class 35
- ❌ Restart policy + Docker enabled at boot — *fix:* restart: unless-stopped; systemctl enable docker; test with a real reboot
- ❌ Log rotation — *fix:* logging: json-file, max-size 10m, max-file 3
- ❌ Backup of irreplaceable volumes, restore tested — *fix:* tar the volume off-VM; practise a restore
- ❌ GPU actually used (GPU VM only) — *fix:* apply docker-compose.gpu.yml; nvidia-smi inside the container
- ❌ External uptime check + metrics — *fix:* uptime probe from outside the VM; scrape /metrics (Week 15)

**after the checklist: 12/12**

- ✅ Only 22/80/443 reachable from the internet
- ✅ HTTPS via a reverse proxy
- ✅ Login in front of the UI (health data!)
- ✅ SSH key-only
- ✅ Secrets only in a server-side .env (chmod 600)
- ✅ Every image tag pinned
- ✅ Healthcheck on every long-running service
- ✅ Restart policy + Docker enabled at boot
- ✅ Log rotation
- ✅ Backup of irreplaceable volumes, restore tested
- ✅ GPU actually used (GPU VM only)
- ✅ External uptime check + metrics

**পড়ার নিয়ম:** প্রথম চেষ্টায় stack "চলে" — কিন্তু Ollama (auth ছাড়া) আর API internet-এ খোলা, HTTPS নেই, login নেই, VM reboot হলে UI ফেরে না, log disk ভরায়, backup কখনো restore করে দেখা হয়নি, আর GPU VM-এর টাকা দিয়ে model চলছে CPU-তে। প্রতিটা ❌ একটা ছোট config পরিবর্তন — কিন্তু কোনোটাই `docker compose up` নিজে বলে দেয় না।

Checklist-এর একটা অংশ (`public_ports`, tag, healthcheck) compose file থেকেই যাচাই হয়; বাকিটা (`https`, `restore_tested`, `ollama_processor`) মানুষকে VM-এ গিয়ে দেখতে হয়। দুটো ভাগ আলাদা রাখুন — automate যা করা যায়, বাকিটা release-এর আগে হাতে tick।

Reverse proxy-র পুরো config এতটুকুই (Caddy নিজে certificate নেয় আর renew করে; domain-এর DNS VM-এ point করা থাকতে হবে, port 80/443 খোলা):

```caddyfile
assistant.example.com {
    basic_auth {
        demo <bcrypt-hash-from: caddy hash-password>
    }
    reverse_proxy ui:8501
}
```

## 📝 Key Takeaways

* **Driver host-এ, CUDA library image-এ, সেতু Container Toolkit।** Container GPU পায় শুধু চাইলে: `--gpus all` বা compose-এর `deploy.resources.reservations.devices` — আলাদা `docker-compose.gpu.yml`-এ, যাতে CPU machine-এ base file ভাঙে না।
* **VRAM = weights + KV cache + overhead।** llama3.1:8b Q4_K_M @৮k ≈ ৬-৭ GB (estimate); KV প্রতি token ১২৮ KiB — লম্বা context-এর দাম। না আঁটলে নীরব partial offload।
* **GPU-র ব্যর্থতা নীরব।** প্রমাণ: container-এর ভেতরে `nvidia-smi`, আর `ollama ps`-এ `100% GPU`।
* **Cloud bill-এর বেশিরভাগ অলস ঘণ্টা।** Right-size (৮B-এ A100 না), schedule, আর traffic কম হলে breakeven হিসাব — কিন্তু health data-র সিদ্ধান্ত শুধু টাকার না।
* **"চলে" ≠ "deployed"।** শুধু 80/443, HTTPS + login, secret server-এর `.env`-এ, pinned tag (rollback), reboot-এ ফেরে, log rotation, restore-tested backup, বাইরে থেকে monitoring।

## 🧠 Brain Teasers

1. Section 1-এ `llama3.1:8b` Q4_K_M, context ১৬k, KV q8_0 — ৮ GB GPU-তে আঁটে? f16 KV-তে?
2. `OVERHEAD_GB` ০.৮ থেকে ১.৫ করুন। `fit` table-এর কোন cell বদলায়? নিজের GPU-তে overhead আসলে কত, কীভাবে মাপবেন?
3. Section 3-এ নিজের পছন্দের দুই provider-এর আসল দাম বসান (তারিখসহ লিখে রাখুন)। ২৪/৭ বনাম office hours-এ পার্থক্য কত? Breakeven দিনে কয়টা প্রশ্ন?
4. `api_month`-এ `in_tokens` ১৫০০ থেকে ৬০০ করুন (top-k কমিয়ে)। Bill কত কমে? Answer quality-তে এর ঝুঁকি কী?
5. `CHECKS`-এ একটা নতুন item যোগ করুন: "Chroma আর Ollama-র কোনো port public না" — `public_ports` থেকে আলাদা কীভাবে লিখবেন যাতে কোন service দোষী সেটাও বলে?